In [ ]:
"""
Colab KTO dataset builder
Builds a KTO dataset from DPO pairs (prompt/chosen/rejected).
Input: finetuning_data/crm-dpo-dataset/cycle_01_v4.jsonl
Output: finetuning_data/crm-kto-dataset/cycle_01.jsonl
"""


In [ ]:
import torch
torch.cuda.is_available()

import os
print(os.getcwd())
print(os.listdir())


In [ ]:
!pip install datasets peft trl bitsandbytes accelerate
!pip install -U transformers
!pip show transformers


In [ ]:
!git clone https://github.com/jjjh02/AmoRe_crm_generator.git
%cd AmoRe_crm_generator
!git checkout jinhyeok


In [ ]:
os.chdir("/content/AmoRe_crm_generator/finetuning")
print(os.getcwd())


In [ ]:
from dotenv import load_dotenv
load_dotenv()


In [2]:
import json
import os
import random

DATASET_DIR = "./finetuning_data"
INPUT_PATH = os.path.join(DATASET_DIR, "crm-dpo-dataset", "cycle_01_v4.jsonl")
OUTPUT_DIR = os.path.join(DATASET_DIR, "crm-kto-dataset")
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "cycle_01.jsonl")

# KTO output schema options:
# - completion_label: {prompt, completion, label} with label 1 (chosen), 0 (rejected)
# - chosen_rejected: {prompt, chosen, rejected} (kept as DPO-style pairs)
KTO_FORMAT = "completion_label"

# Desired negative ratio for completion_label format (label=0).
NEGATIVE_RATIO = 0.70
RANDOM_SEED = 42


def _load_dpo_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Input not found: {path}")
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            yield json.loads(line)


def _iter_kto_records(row, fmt):
    prompt = row.get("prompt", "")
    chosen = row.get("chosen", "")
    rejected = row.get("rejected", "")
    if fmt == "completion_label":
        yield {"prompt": prompt, "completion": chosen, "label": 1}
        yield {"prompt": prompt, "completion": rejected, "label": 0}
    elif fmt == "chosen_rejected":
        yield {"prompt": prompt, "chosen": chosen, "rejected": rejected}
    else:
        raise ValueError(f"Unknown KTO_FORMAT: {fmt}")


def _select_label(label, neg_ratio):
    if label not in (0, 1):
        return True
    if label == 0:
        return random.random() < neg_ratio
    return random.random() < (1.0 - neg_ratio)


os.makedirs(OUTPUT_DIR, exist_ok=True)
random.seed(RANDOM_SEED)

count_in = 0
count_out = 0
count_pos = 0
count_neg = 0

with open(OUTPUT_PATH, "w", encoding="utf-8") as f_out:
    for row in _load_dpo_jsonl(INPUT_PATH):
        count_in += 1
        for record in _iter_kto_records(row, KTO_FORMAT):
            if KTO_FORMAT == "completion_label":
                label = record.get("label")
                if not _select_label(label, NEGATIVE_RATIO):
                    continue
                if label == 0:
                    count_neg += 1
                elif label == 1:
                    count_pos += 1
            f_out.write(json.dumps(record, ensure_ascii=False) + "\n")
            count_out += 1

print(f"Input records: {count_in}")
print(f"Output records: {count_out}")
if KTO_FORMAT == "completion_label":
    total = count_pos + count_neg
    ratio = (count_neg / total) if total else 0.0
    print(f"Label counts: pos={count_pos} neg={count_neg} neg_ratio={ratio:.2f}")
print(f"Output path: {OUTPUT_PATH}")


Input records: 1300
Output records: 1292
Label counts: pos=378 neg=914 neg_ratio=0.71
Output path: ./finetuning_data\crm-kto-dataset\cycle_01.jsonl
